- **Nama:** Reza Fathia
- **NIM:** H061251068
- **Program Studi:** Geofisika (B)
- **Fakultas:** Fakultas Matematika dan Ilmu Pengetahuan Alam (FMIPA)
- **Universitas:** Universitas Hasanuddin
- **Mata Kuliah:** Kecerdasan Artifisial untuk Geofisika
---
# RECIPE R-IO-001: AUDIT KUALITAS DATA STASIUN METEOROLOGI MENGGUNAKAN PYTHON
**Mata Kuliah:** Geofisika  
**Bidang:** Pengolahan Data Ilmu Kebumian  
**Recipe ID:** R-IO-001  
**Prompt ID:** P-IO-001  
**Status:** Not yet executed (belum dijalankan pada data aktual)
---
## 1. Problem (Permasalahan)
Data meteorologi sering digunakan dalam penelitian geofisika, terutama untuk mempelajari kondisi atmosfer, curah hujan, perubahan suhu, dan fenomena iklim. Sebelum data tersebut digunakan untuk analisis lebih lanjut, kualitas data perlu diperiksa agar kesalahan pada tabel tidak memengaruhi hasil penelitian.

Recipe ini bertujuan melakukan audit kualitas data atau *Quality Assurance/Quality Control* (QA/QC) terhadap data stasiun meteorologi dalam format CSV. Pemeriksaan difokuskan pada struktur kolom, rentang waktu, nilai yang hilang (*missing values*), duplikasi waktu pengamatan, serta ringkasan statistik variabel numerik.

Hasil akhir berupa laporan pemeriksaan kualitas data tanpa mengubah atau menghapus data asli.

## 2. Minimal Theory (Teori Dasar)
Audit kualitas data merupakan proses pemeriksaan untuk mengetahui apakah data memiliki struktur dan kelengkapan yang sesuai sebelum digunakan dalam analisis ilmiah.

Aspek yang diperiksa meliputi:
1. **Schema data:** Memastikan kolom yang diperlukan tersedia dan memiliki tipe data yang sesuai.
2. **Missing values:** Mengetahui jumlah nilai kosong pada setiap kolom.
3. **Duplicate timestamp:** Mengidentifikasi waktu pengamatan yang tercatat lebih dari satu kali.
4. **Rentang waktu:** Menentukan tanggal pengamatan paling awal dan paling akhir.
5. **Statistik deskriptif:** Menghitung nilai minimum, maksimum, rata-rata, dan simpangan baku untuk variabel numerik.

Nilai yang hilang dapat memengaruhi perhitungan statistik, sedangkan duplikasi waktu dapat menyebabkan kesalahan ketika data diagregasi menjadi data harian atau bulanan.

Pemeriksaan ini tidak secara otomatis membuktikan bahwa seluruh nilai pengamatan benar secara fisis. Data yang lolos pemeriksaan struktur tetap perlu ditinjau berdasarkan satuan, rentang nilai yang masuk akal, metadata, dan sumber pengamatan.

## 3. Input dan Batasan

### Input

- Satu file CSV data stasiun meteorologi.
- Kolom waktu bernama `time`.
- Kolom lain dapat berisi variabel meteorologi, seperti suhu, kelembapan, kecepatan angin, atau curah hujan.

### Batasan

- File asli hanya dibaca dan tidak boleh ditimpa.
- Data yang kosong tidak diisi secara otomatis.
- Baris duplikat tidak dihapus.
- Satuan data tidak diasumsikan apabila tidak tersedia dalam metadata.
- Pemeriksaan dibatasi pada kualitas struktur dan ringkasan statistik, bukan prediksi cuaca atau perhitungan indeks iklim.

## 4. Prompt (Instruksi untuk AI)

Buatlah program Python untuk melakukan audit kualitas data stasiun meteorologi dari file CSV. Program harus membaca file tanpa mengubah data asli, memeriksa struktur kolom, mengidentifikasi nilai kosong, mendeteksi duplikasi waktu pengamatan, menentukan rentang waktu, serta menghitung statistik deskriptif variabel numerik.

Simpan hasil audit sebagai laporan Markdown. Sertakan jumlah baris dan kolom, jumlah nilai kosong pada setiap kolom, jumlah timestamp duplikat, rentang waktu pengamatan, dan statistik numerik.

Jika kolom `time` tidak tersedia atau format waktunya tidak dapat dibaca, tampilkan peringatan yang jelas. Jangan melakukan imputasi, penghapusan baris, atau mengklaim bahwa program telah berhasil dijalankan sebelum hasil eksekusinya diperiksa.

## 5. Code (Implementasi Python)
```python
import pandas as pd
from pathlib import Path

# 1. Menentukan lokasi file
input_file = Path("station-example.csv")
output_file = Path("station-summary.md")

# 2. Memeriksa keberadaan file
if not input_file.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {input_file}"
    )

# 3. Membaca data tanpa mengubah file asli
data = pd.read_csv(input_file)

# 4. Memeriksa struktur data
report = []
report.append("# Laporan Audit Data Meteorologi\n")
report.append(f"Jumlah baris: {len(data)}")
report.append(f"Jumlah kolom: {len(data.columns)}")
report.append(
    "Nama kolom: " + ", ".join(data.columns)
)

# 5. Memeriksa nilai kosong
report.append("\n## Pemeriksaan Missing Values")
missing = data.isna().sum()

for column, count in missing.items():
    report.append(f"- {column}: {count} nilai kosong")

# 6. Memeriksa kolom waktu
report.append("\n## Pemeriksaan Waktu")

if "time" in data.columns:
    parsed_time = pd.to_datetime(
        data["time"], errors="coerce"
    )

    invalid_time = parsed_time.isna().sum()

    report.append(
        f"Jumlah waktu tidak valid: {invalid_time}"
    )

    valid_time = parsed_time.dropna()

    if not valid_time.empty:
        report.append(
            f"Waktu awal: {valid_time.min()}"
        )
        report.append(
            f"Waktu akhir: {valid_time.max()}"
        )

    duplicate_count = (
        parsed_time.notna()
        & parsed_time.duplicated(keep=False)
    ).sum()

    report.append(
        "Jumlah baris dengan timestamp duplikat: "
        f"{duplicate_count}"
    )
else:
    report.append(
        "PERINGATAN: Kolom 'time' tidak ditemukan."
    )

# 7. Statistik deskriptif
report.append("\n## Statistik Deskriptif")

numeric_data = data.select_dtypes(
    include="number"
)

if not numeric_data.empty:
    report.append(
        numeric_data.describe().to_markdown()
    )
else:
    report.append(
        "Tidak ditemukan kolom numerik."
    )

# 8. Menyimpan laporan
output_file.write_text(
    "\n".join(report),
    encoding="utf-8"
)

print("Audit selesai.")
print(f"Laporan disimpan di: {output_file}")
```

### Dependencies

Program memerlukan pustaka `pandas` dan `tabulate`. Jika belum tersedia, jalankan perintah berikut di terminal:

```bash
pip install pandas tabulate
```

Nama file input dan output dapat disesuaikan dengan dataset yang digunakan.

## 6. Result (Hasil yang Diharapkan)

Program dirancang menghasilkan file `station-summary.md` yang memuat:

1. Jumlah baris dan kolom pada dataset.
2. Daftar nama kolom yang tersedia.
3. Jumlah nilai kosong pada setiap kolom.
4. Rentang waktu pengamatan.
5. Jumlah baris dengan timestamp duplikat.
6. Statistik deskriptif untuk variabel numerik.

Hasil tersebut digunakan untuk menilai apakah data memerlukan pemeriksaan lanjutan sebelum dipakai dalam penelitian.

**Status hasil:** Hasil aktual belum tersedia karena program belum dijalankan pada dataset pengguna. Oleh karena itu, angka pemeriksaan dan kesimpulan kualitas data belum dapat ditetapkan.

## 7. Optional Check (Pemeriksaan Tambahan)

Setelah program dijalankan, beberapa pemeriksaan tambahan yang dapat dilakukan adalah:

1. Memastikan laporan berhasil dibuat dan tidak kosong.
2. Membandingkan jumlah baris pada laporan dengan jumlah baris file asli.
3. Memeriksa apakah format tanggal dan waktu sudah sesuai.
4. Meninjau nilai minimum dan maksimum berdasarkan satuan serta karakteristik fisis variabel.
5. Memastikan file CSV asli tidak berubah.
6. Mencatat versi Python, pustaka yang digunakan, lokasi input dan output, serta status eksekusi program.

Jika ditemukan nilai kosong atau timestamp duplikat, hasil audit perlu ditinjau terlebih dahulu sebelum menentukan tindakan perbaikan. Data asli tetap dipertahankan agar proses pemeriksaan dapat ditelusuri kembali.

## 8. Kesimpulan

Recipe R-IO-001 merupakan prosedur yang dapat digunakan kembali untuk memeriksa kualitas awal data stasiun meteorologi sebelum dilakukan analisis lebih lanjut dalam bidang geofisika dan ilmu iklim. Dengan menggunakan Python, pemeriksaan struktur, kelengkapan data, waktu pengamatan, dan statistik deskriptif dapat dilakukan secara sistematis.

Recipe yang baik tidak hanya berisi instruksi atau kode, tetapi juga menjelaskan tujuan, teori dasar, input, batasan, keluaran, serta pemeriksaan hasil. Keberhasilan prosedur harus didasarkan pada bukti eksekusi dan pemeriksaan keluaran, bukan hanya karena kode telah ditulis.